# 08. 딥러닝 회귀 - 실무전용 문제 모음

> **실무 전용 노트북** - 이론 설명 없이 TODO 문제만 모았습니다. (관련 이론 노트북: 08_딥러닝_회귀.ipynb)
이미 개념은 이해했다는 전제로, 손으로 더 많이 풀어보며 완전히 몸에 익히는 것이 목표입니다. 정답을 먼저 보지 말고 반드시 스스로 코드를 작성해본 뒤 펼쳐서 비교하세요.

이론 노트북에서는 Titanic Fare 예측으로 연습했다면, 여기서는 **매장 매출 예측(retail_sales) 데이터**로 딥러닝 회귀를 다시 연습합니다.

## 목차
1. 모델 설계 & 컴파일 연습
2. 모델 학습 연습
3. 성능 시각화 연습

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

tf.random.set_seed(100)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sales = pd.read_csv('../data/retail_sales.csv')
sales['광고비_만원'] = sales['광고비_만원'].fillna(sales['광고비_만원'].mean())
q1, q3 = sales['매출_만원'].quantile([0.25, 0.75])
iqr = q3 - q1
sales['매출_만원'] = sales['매출_만원'].clip(upper=q3 + 1.5 * iqr)
sales_enc = pd.get_dummies(sales, columns=['지역', '주말여부'], drop_first=True)
X = sales_enc.drop(columns=['매출_만원'])
y = sales_enc['매출_만원']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_train.shape

## 1. 모델 설계 & 컴파일 연습

**문제 1.** 입력층 32(relu)→16(relu)→출력 1(활성화 없음)인 `model`을 만들고 `loss='mse'`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train_s.shape[1],)))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))  # 회귀 출력층: 노드 1개, activation 없음
model.compile(optimizer='adam', loss='mse', metrics=['mae'])  # 회귀는 loss='mse'가 기본
model.summary()
```

</details>

## 2. 모델 학습 연습

**문제 2.** `EarlyStopping(patience=5)`를 사용해 `epochs=50`으로 학습시키세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)  # patience=5: 5번 연속 개선이 없으면 학습 중단
history = model.fit(X_train_s, y_train, epochs=50, batch_size=16, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
print(len(history.history['loss']))  # epochs=50을 다 채우지 못하고 조기 종료됐다면 이 길이가 50보다 작음
```

</details>

## 3. 성능 시각화 연습

**문제 3.** loss curve를 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')  # 두 곡선이 함께 내려가면 정상, val만 다시 올라가면 과적합 신호
plt.legend(); plt.show()
```

</details>

**문제 4.** 예측 vs 실제 산점도를 그리고 R2를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import r2_score
pred = model.predict(X_test_s).flatten()  # predict 결과가 (샘플수,1) 형태로 나오므로 flatten()으로 1차원으로 펼침
plt.scatter(y_test, pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # y=x 대각선에 가까울수록 예측이 정확함
plt.show()
print(r2_score(y_test, pred))
```

</details>